# Slater-damped multipoles vs frozen-density electrostatics

Per-geometry fits (`rsfff.qcgen.elst_fit`) of the force field's own electrostatics form -- point nucleus *Z* + Slater 1s shell (*b*) carrying `[q - Z, mu, Theta]` -- to the reference potential and field of each monomer (`rsfff.qcgen.elst_db`, wB97M-V/def2-TZVPD, 0.6-3.0x vdW grids). Pair energies then test the two-centre terms the grids never see.

Fits are produced by `scripts/elst_fit.py` (minutes per 100 monomers); this notebook only loads and analyses.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ase.io import read
from rsfff.qcgen import elst_db as D, elst_fit as F

ROOT = Path('~/research/rsfff/data/elst_db').expanduser()
db = D.MonomerDB(ROOT)
fits = {m: F.load_fits(ROOT / 'fits' / f'fits_{m}.json') for m in ('point_r2', 'slater_r2')}

## One-body: potential on the grids (median over conformers, kcal/mol/e)

In [ ]:
for m, f in fits.items():
    print(m); display(pd.DataFrame(F.grid_error_table(db, f)).set_index('group').round(3))

## Two-body: pair energies from the per-geometry parameters (rotated into place)

In [ ]:
sets = {s: read(str(ROOT / f'{s}.extxyz'), ':') for s in ('pairs_neutral', 'pairs_ion_neutral', 'pairs_ion_ion')}
rng = np.random.default_rng(0)
sub = {s: [fr[i] for i in rng.choice(len(fr), min(400, len(fr)), replace=False)] for s, fr in sets.items()}
res = {}
for m, f in fits.items():
    for s, fr in sub.items():
        rows, c, err, ref = F.pair_error_table(db, f, fr, penetration=(m != 'point_r2'))
        res[(m, s)] = (c, err, ref)
        print(m, s); display(pd.DataFrame(rows).round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, s in zip(axes, sub):
    for m, col in (('point_r2', '#eb6834'), ('slater_r2', '#2a78d6')):
        c, err, ref = res[(m, s)]
        ax.scatter(c, np.abs(err), s=6, color=col, label=m, alpha=0.6)
    ax.set(yscale='log', xlabel='closest contact / A', title=s)
axes[0].set_ylabel('|E_model - E_ref| / kcal/mol'); axes[0].legend()
plt.tight_layout()

## How the parameters move with geometry (gauge vs. physics)

In [ ]:
for sp in ('water', 'hf', 'nh3'):
    print(sp); display(pd.DataFrame(F.parameter_spread(db, fits['slater_r2'], sp)).set_index('atom').round(3))